# Evaluate HelpSteer2 All-Method Adapter Merges

This notebook runs the full HelpSteer2 all-method merge evaluation. It uses the coefficient table, the fixed prompt file, and the local HelpSteer2 adapters to generate responses, compute proxy scores, and summarize preference-weighted utility.

Generated answers and result summaries are saved under `results/`. Adapter folders, zip files, safetensors, bin files, checkpoints, and model weights should not be committed to GitHub.


## 1. Clone or update repository


In [ ]:
from pathlib import Path
import os
import subprocess

repo_path = Path("/content/master-thesis")
repo_url = "https://github.com/NZhang137/master-thesis.git"

os.chdir("/content")

if (repo_path / ".git").is_dir():
    print("Repository found. Pulling the latest changes...")
    subprocess.run(["git", "-C", str(repo_path), "pull"], check=True)
elif repo_path.exists():
    raise RuntimeError(
        f"{repo_path} exists but is not a Git repository. "
        "Rename or remove it, then run this cell again."
    )
else:
    print("Cloning the repository...")
    subprocess.run(["git", "clone", repo_url, str(repo_path)], check=True)

os.chdir(repo_path)
print(f"Current folder: {Path.cwd()}")


## 2. Show repository structure


In [ ]:
!pwd
!ls
!ls scripts
!ls data/evaluation_prompts
!ls results || echo "No results folder found yet."


## 3. Check GPU

A GPU is strongly recommended because this run evaluates every coefficient row on every fixed prompt.


In [ ]:
!nvidia-smi


## 4. Install dependencies

This pins pandas and numpy to Colab-compatible versions to avoid common resolver conflicts. If you later see a `torchao` version error, run `!pip install -q -U torchao` and restart the runtime.


In [ ]:
!pip install -q "pandas==2.2.2" "numpy<2.1" transformers datasets peft accelerate safetensors


## 5. Upload or unzip local adapters

If the `adapters/` folder is missing, upload your local `helpsteer2_adapters_longrun.zip` backup and unzip it. The zip is only a local backup and should not be committed.


In [ ]:
!ls adapters || echo "No adapters folder found yet."


In [ ]:
from pathlib import Path
from google.colab import files

adapter_paths = [
    Path("adapters/helpsteer2-gpt2-helpfulness-adapter"),
    Path("adapters/helpsteer2-gpt2-correctness-adapter"),
    Path("adapters/helpsteer2-gpt2-coherence-adapter"),
    Path("adapters/helpsteer2-gpt2-complexity-adapter"),
    Path("adapters/helpsteer2-gpt2-verbosity-adapter"),
]

if all(path.is_dir() for path in adapter_paths):
    print("All HelpSteer2 adapter folders are already present.")
else:
    print("Upload helpsteer2_adapters_longrun.zip now.")
    uploaded = files.upload()


In [ ]:
!if [ -f helpsteer2_adapters_longrun.zip ]; then unzip -o helpsteer2_adapters_longrun.zip; else echo "No helpsteer2_adapters_longrun.zip found, skipping unzip."; fi
!ls adapters || echo "No adapters folder found."


## 6. Verify adapters and input files


In [ ]:
!python scripts/check_helpsteer2_adapters.py


In [ ]:
from pathlib import Path

required_files = [
    Path("results/helpsteer2_all_method_coefficients.csv"),
    Path("results/helpsteer2_method_costs.csv"),
    Path("data/evaluation_prompts/helpsteer2_fixed_prompts.jsonl"),
]

for file_path in required_files:
    if not file_path.is_file():
        raise FileNotFoundError(f"Missing required file: {file_path}")
    print(f"Found: {file_path}")


## 7. Run full all-method evaluation

This command evaluates every coefficient row on every fixed prompt. It can take a while because it repeatedly loads merged adapters and generates responses.


In [ ]:
!python scripts/evaluate_helpsteer2_all_method_merges.py


## 8. Inspect generated outputs


In [ ]:
!ls results | grep helpsteer2_all_method


In [ ]:
import pandas as pd

generations = pd.read_csv("results/helpsteer2_all_method_generations.csv")
scores = pd.read_csv("results/helpsteer2_all_method_scores.csv")
summary = pd.read_csv("results/helpsteer2_all_method_result_summary.csv")

print(f"Generations rows: {len(generations)}")
display(generations.head())

print(f"Scores rows: {len(scores)}")
display(scores.head())

print(f"Summary rows: {len(summary)}")
display(summary.head())


## 9. Best method per preference


In [ ]:
best_by_preference = (
    summary.sort_values("mean_utility", ascending=False)
    .groupby("preference_name", as_index=False)
    .first()[[
        "preference_name",
        "method",
        "hyperparameter_id",
        "mean_utility",
        "improvement_over_direct_preference",
        "improvement_over_uniform",
    ]]
)
display(best_by_preference)


## 10. Read the Markdown and JSON summaries


In [ ]:
!cat results/helpsteer2_all_method_result_summary.md


In [ ]:
import json

with open("results/helpsteer2_all_method_result_summary.json", "r", encoding="utf-8") as f:
    result_metadata = json.load(f)

print(json.dumps(result_metadata, indent=2)[:3000])


## 11. Git safety check

It is okay to commit small CSV, JSON, and Markdown result files if useful. Do not commit `adapters/`, zip files, safetensors, bin files, checkpoints, or model weights.


In [ ]:
!git status
